# ResNet-18 — Urban Morphology Classification

Requirements covered:
1. Class-weighted loss (radial imbalance)
2. Dataset-size experiments (25 / 50 / 75 / 100 %)
3. Hyperparameter sweep (2 LRs × 2 batch sizes)
4. Per-class accuracy / precision / recall / F1 + confusion matrix
5. Shared `dataset/train`, `dataset/val`, `dataset/test` splits
6. Best-checkpoint saving

## 1 — Imports & device

In [ ]:
import os, copy, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models, datasets, transforms
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('PyTorch version :', torch.__version__)
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Device          :', DEVICE)

## 2 — Paths & transforms

In [ ]:
DATA_DIR  = os.path.join(os.path.dirname(os.getcwd()), 'dataset')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'val')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
MODELS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),          # satellite imagery is orientation-free
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Full datasets (transforms applied at __getitem__ so we can subset freely)
full_train_data = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_data        = datasets.ImageFolder(VAL_DIR,   transform=eval_transform)
test_data       = datasets.ImageFolder(TEST_DIR,  transform=eval_transform)

CLASS_NAMES = full_train_data.classes   # sorted alphabetically: ['grid', 'organic', 'radial']
NUM_CLASSES  = len(CLASS_NAMES)

print('Classes    :', CLASS_NAMES)
print('Train size :', len(full_train_data))
print('Val   size :', len(val_data))
print('Test  size :', len(test_data))
print('\nIMPORTANT — class index order:', {c: i for i, c in enumerate(CLASS_NAMES)})

## 3 — Class weights (requirement 1)

ImageFolder sorts **alphabetically**, so class indices are:
- 0 → grid  (211 tiles)
- 1 → organic (200 tiles)
- 2 → radial  (134 tiles  ← minority)

Weight formula: `min_count / class_count`  
Minority class gets weight 1.0; majority classes get fractional weights.

In [ ]:
# Counts: grid=211, organic=200, radial=134
# Alphabetical order matches ImageFolder indices 0, 1, 2
CLASS_WEIGHTS = torch.tensor([134/211, 134/200, 1.0], dtype=torch.float).to(DEVICE)
print('Class weights:', dict(zip(CLASS_NAMES, CLASS_WEIGHTS.cpu().tolist())))

## 4 — Helper functions

In [ ]:
def build_resnet18(num_classes, device):
    """Load pretrained ResNet-18, freeze backbone, replace head."""
    import ssl, certifi, urllib.request
    ssl_ctx = ssl.create_default_context(cafile=certifi.where())
    urllib.request.install_opener(urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_ctx)))

    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in m.parameters():
        param.requires_grad = False
    for param in m.layer4.parameters():
        param.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += outputs.max(1)[1].eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total


def get_subset(dataset, fraction, seed=42):
    """Return a deterministic random subset of `fraction` of `dataset`."""
    n = int(len(dataset) * fraction)
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=n, replace=False).tolist()
    return Subset(dataset, indices)


def run_test_eval(model, loader, device, class_names):
    """Return predictions, labels, and print full metrics."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds  = model(images).max(1)[1].cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def plot_confusion(labels, preds, class_names, title):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',  xticklabels=class_names,
                yticklabels=class_names, ax=axes[0], linewidths=0.5)
    axes[0].set(title='Confusion Matrix — Counts', xlabel='Predicted', ylabel='Actual')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', xticklabels=class_names,
                yticklabels=class_names, ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
    axes[1].set(title='Normalised (Recall)', xlabel='Predicted', ylabel='Actual')
    plt.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

print('Helper functions defined.')

## 5 — Hyperparameter sweep (requirement 3)

Grid: **2 learning rates × 2 batch sizes = 4 combinations**  
Each trained for 10 epochs on the full training set — quick comparison pass.

In [ ]:
HP_EPOCHS   = 10
LEARNING_RATES = [1e-3, 1e-4]
BATCH_SIZES    = [16, 32]

val_loader_hp = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=0)
hp_results = []

for lr in LEARNING_RATES:
    for bs in BATCH_SIZES:
        print(f'\n--- LR={lr}  BS={bs} ---')
        train_loader_hp = DataLoader(
            full_train_data, batch_size=bs, shuffle=True, num_workers=0
        )
        model     = build_resnet18(NUM_CLASSES, DEVICE)
        criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=1e-4
        )
        scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

        best_val = 0.0
        for epoch in range(HP_EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader_hp, criterion, optimizer, DEVICE)
            val_loss,   val_acc   = eval_epoch(model, val_loader_hp, criterion, DEVICE)
            scheduler.step()
            if val_acc > best_val:
                best_val = val_acc
            print(f'  Ep {epoch+1:02d}/{HP_EPOCHS}  '
                  f'Train {train_loss:.4f}/{train_acc:.4f}  '
                  f'Val {val_loss:.4f}/{val_acc:.4f}')

        hp_results.append({'lr': lr, 'bs': bs, 'best_val_acc': best_val})
        print(f'  => Best val acc: {best_val:.4f}')

print('\n=== Hyperparameter Sweep Summary ===')
for r in hp_results:
    print(f"LR={r['lr']:<6}  BS={r['bs']:<3}  Best Val Acc={r['best_val_acc']:.4f}")

best_hp = max(hp_results, key=lambda x: x['best_val_acc'])
BEST_LR = best_hp['lr']
BEST_BS = best_hp['bs']
print(f'\nSelected: LR={BEST_LR}  BS={BEST_BS}')

In [ ]:
# Plot hyperparameter sweep results
fig, ax = plt.subplots(figsize=(8, 5))
labels_hp = [f"LR={r['lr']}\nBS={r['bs']}" for r in hp_results]
accs_hp   = [r['best_val_acc'] * 100 for r in hp_results]
bars = ax.bar(labels_hp, accs_hp, color=['steelblue', 'darkorange', 'seagreen', 'crimson'],
              edgecolor='white', width=0.5)
for bar, acc in zip(bars, accs_hp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', fontsize=10)
ax.set(title='ResNet-18 — Hyperparameter Sweep (10 epochs)',
       ylabel='Best Val Accuracy (%)', ylim=(0, 110))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6 — Dataset-size experiments (requirement 2)

Using best hyperparameters from sweep.  
Fractions: **25%, 50%, 75%, 100%** of training data.

In [ ]:
FRACTIONS  = [0.25, 0.50, 0.75, 1.00]
DS_EPOCHS  = 15
val_loader_ds = DataLoader(val_data, batch_size=BEST_BS, shuffle=False, num_workers=0)

ds_results = []

for frac in FRACTIONS:
    subset   = get_subset(full_train_data, frac)
    n_subset = len(subset)
    loader   = DataLoader(subset, batch_size=BEST_BS, shuffle=True, num_workers=0)

    print(f'\n--- Fraction={frac*100:.0f}%  ({n_subset} samples) ---')
    model     = build_resnet18(NUM_CLASSES, DEVICE)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=BEST_LR, weight_decay=1e-4
    )
    scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

    best_val  = 0.0
    val_curve = []
    for epoch in range(DS_EPOCHS):
        train_one_epoch(model, loader, criterion, optimizer, DEVICE)
        _, val_acc = eval_epoch(model, val_loader_ds, criterion, DEVICE)
        scheduler.step()
        val_curve.append(val_acc)
        if val_acc > best_val:
            best_val = val_acc

    ds_results.append({'frac': frac, 'n': n_subset,
                       'best_val_acc': best_val, 'curve': val_curve})
    print(f'  Best val acc: {best_val:.4f}')

print('\n=== Dataset-Size Summary ===')
for r in ds_results:
    print(f"  {r['frac']*100:.0f}%  (n={r['n']})  Best Val Acc={r['best_val_acc']:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves per fraction
for r in ds_results:
    ax1.plot(range(1, DS_EPOCHS + 1),
             [a * 100 for a in r['curve']],
             marker='o', label=f"{r['frac']*100:.0f}% ({r['n']} imgs)")
ax1.set(title='Val Accuracy by Dataset Size', xlabel='Epoch', ylabel='Val Acc (%)')
ax1.legend(); ax1.grid(alpha=0.3)

# Bar: best val acc vs fraction
fracs  = [r['frac'] * 100 for r in ds_results]
bst    = [r['best_val_acc'] * 100 for r in ds_results]
bars2  = ax2.bar([f"{f:.0f}%" for f in fracs], bst,
                 color='steelblue', edgecolor='white', width=0.5)
for bar, acc in zip(bars2, bst):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', fontsize=10)
ax2.set(title='Best Val Accuracy vs Training Set Size',
        xlabel='Training fraction', ylabel='Best Val Acc (%)', ylim=(0, 110))
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('ResNet-18 — Dataset Size Experiments', fontsize=13)
plt.tight_layout(); plt.show()

## 7 — Full training with best config (requirement 6)

Train on 100% of training data with the best LR and batch size.  
Saves best checkpoint to `models/best_resnet18.pth`.

In [ ]:
NUM_EPOCHS   = 20
SAVE_PATH    = os.path.join(MODELS_DIR, 'best_resnet18.pth')

train_loader = DataLoader(full_train_data, batch_size=BEST_BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_data,        batch_size=BEST_BS, shuffle=False, num_workers=0)

model     = build_resnet18(NUM_CLASSES, DEVICE)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=BEST_LR, weight_decay=1e-4
)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_state   = None

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss);     history['val_acc'].append(val_acc)

    tag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = copy.deepcopy(model.state_dict())
        torch.save({'epoch': epoch + 1,
                    'model_state': best_state,
                    'optimizer': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'classes': CLASS_NAMES,
                    'best_lr': BEST_LR,
                    'best_bs': BEST_BS}, SAVE_PATH)
        tag = ' ← best'

    print(f'Epoch [{epoch+1:02d}/{NUM_EPOCHS}]  '
          f'Train {train_loss:.4f}/{train_acc:.4f}  '
          f'Val {val_loss:.4f}/{val_acc:.4f}{tag}')

print(f'\nDone. Best val acc: {best_val_acc:.4f}  →  {SAVE_PATH}')

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train')
ax1.plot(epochs_range, history['val_loss'],   's-', label='Val')
ax1.set(title='Loss per Epoch', xlabel='Epoch', ylabel='CrossEntropy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history['train_acc']], 'o-', label='Train')
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   's-', label='Val')
ax2.set(title='Accuracy per Epoch', xlabel='Epoch', ylabel='Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f'ResNet-18 — Full Training (LR={BEST_LR}, BS={BEST_BS})', fontsize=13)
plt.tight_layout(); plt.show()

## 8 — Test-set evaluation (requirement 4)

Load best checkpoint and report per-class accuracy, precision, recall, F1, and confusion matrix.

In [ ]:
# Load best checkpoint
ckpt       = torch.load(SAVE_PATH, map_location=DEVICE)
best_model = build_resnet18(NUM_CLASSES, DEVICE)
best_model.load_state_dict(ckpt['model_state'])
print(f'Loaded checkpoint — epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f}')

test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0)
preds, labels = run_test_eval(best_model, test_loader, DEVICE, CLASS_NAMES)

test_acc = accuracy_score(labels, preds)
print(f'\nTest Accuracy: {test_acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
plot_confusion(labels, preds, CLASS_NAMES,
               f'ResNet-18 — Test Acc: {test_acc*100:.2f}%')